In [ ]:
%reload_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')
from IPython.display import clear_output

import pandas as pd
from functools import reduce
from tqdm.auto import tqdm
from pandarallel import pandarallel
pandarallel.initialize()

from rdkit import Chem
from rdkit.Chem import PandasTools
PandasTools.RenderImagesInAllDataFrames(images=True)
from rdkit.Chem.Draw import rdDepictor
rdDepictor.SetPreferCoordGen(True)
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')

from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True
IPythonConsole.molSize = 600, 400
IPythonConsole.drawOptions.addAtomIndices = False

clear_output()

In [184]:
from pathlib import Path
import numpy as np
import cairosvg
from rdkit import Chem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D

def export_consistent_mol_figures(
    mols,
    out_dir='figures',
    prefix='',
    canvas_w=300,
    padding=0.02,
    fixed_font_size=None,
    save_png=False,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if isinstance(mols, dict):
        items = list(mols.items())
    else:
        items = [(None, mol) for mol in mols]

    def _safe_name(name):
        text = str(name).strip()
        if not text:
            return 'mol'
        safe = []
        for ch in text:
            if ch.isalnum() or ch in {'-', '_'}:
                safe.append(ch)
            else:
                safe.append('_')
        return ''.join(safe).strip('_') or 'mol'

    valid = []
    for idx, (name, mol) in enumerate(items, start=1):
        if mol is None:
            continue

        m = Chem.Mol(mol)
        rdDepictor.Compute2DCoords(m)
        conf = m.GetConformer()

        bl = []
        for b in m.GetBonds():
            i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
            pi = conf.GetAtomPosition(i)
            pj = conf.GetAtomPosition(j)
            bl.append(((pi.x - pj.x) ** 2 + (pi.y - pj.y) ** 2) ** 0.5)

        mean_bl = float(np.mean(bl)) if bl else 1.0
        xs = [conf.GetAtomPosition(i).x for i in range(m.GetNumAtoms())]
        ys = [conf.GetAtomPosition(i).y for i in range(m.GetNumAtoms())]
        span = max((max(xs) - min(xs)), (max(ys) - min(ys)), 1e-6)
        label = _safe_name(name) if name is not None else chr(ord('a') + idx - 1)
        valid.append((idx, label, m, mean_bl, span))

    if not valid:
        raise ValueError('No valid molecules to draw')

    target_bl = float(np.median([x[3] for x in valid]))
    scaled_spans = []
    for _, _, _, mean_bl, span in valid:
        factor = target_bl / max(mean_bl, 1e-6)
        scaled_spans.append(span * factor)

    max_scaled_span = max(scaled_spans)
    fixed_scale = (1.0 - 2.0 * padding * canvas_w / 160) / max_scaled_span

    for idx, label, mol, mean_bl, _ in valid:
        suffix = label if label is not None else chr(ord('a') + idx - 1)
        stem = f'{prefix}_{suffix}' if prefix else suffix
        svg_path = out_dir / f'{stem}.svg'

        m = Chem.Mol(mol)
        conf = m.GetConformer()
        factor = target_bl / max(mean_bl, 1e-6)
        for a in range(m.GetNumAtoms()):
            p3 = conf.GetAtomPosition(a)
            conf.SetAtomPosition(a, (p3.x * factor, p3.y * factor, 0.0))

        drawer = rdMolDraw2D.MolDraw2DSVG(canvas_w, canvas_w)
        draw_opts = drawer.drawOptions()
        draw_opts.useBWAtomPalette()
        draw_opts.addAtomIndices = False
        draw_opts.fixedBondLength = 22
        draw_opts.multipleBondOffset = 0.2
        draw_opts.bondLineWidth = 0.12 * fixed_font_size
        draw_opts.padding = padding * canvas_w / 160
        draw_opts.additionalAtomLabelPadding = 0.05
        draw_opts.clearBackground = True
        # draw_opts.setBackgroundColour((1, 1, 1))
        draw_opts.fixedFontSize = fixed_font_size
        draw_opts.fixedScale = fixed_scale


        drawer.DrawMolecule(m)
        drawer.FinishDrawing()
        svg_text = drawer.GetDrawingText()

        svg_path.write_text(svg_text, encoding='utf-8')
        print(f'Saved {str(svg_path)}')

        if save_png:
            png_path = out_dir / f'{stem}.png'
            cairosvg.svg2png(bytestring=svg_text.encode('utf-8'), write_to=str(png_path))
            print(f'Saved {str(png_path)}')


https://2025.igem.wiki/marburg/part-collection
https://www.biorxiv.org/content/biorxiv/early/2022/12/08/2022.12.02.518901.full.pdf#page=8.23
https://www.researchgate.net/figure/Examples-of-novel-peptides-produced-by-combinatorial-biosynthesis-via-engineering-NRPS_fig2_379934271
https://www.nature.com/articles/ncomms15349


In [178]:
smis = {
    'enterobactin': 'C1[C@@H](C(=O)OC[C@@H](C(=O)OC[C@@H](C(=O)O1)NC(=O)C2=C(C(=CC=C2)O)O)NC(=O)C3=C(C(=CC=C3)O)O)NC(=O)C4=C(C(=CC=C4)O)O',
    'salmochelin_s4': 'C1[C@@H](C(=O)OC[C@@H](C(=O)OC[C@@H](C(=O)O1)NC(=O)C2=C(C(=CC(=C2)[C@H]3[C@@H]([C@H]([C@@H]([C@H](O3)CO)O)O)O)O)O)NC(=O)C4=C(C(=CC(=C4)[C@H]5[C@@H]([C@H]([C@@H]([C@H](O5)CO)O)O)O)O)O)NC(=O)C6=C(C(=CC=C6)O)O',
    'cyclic_trichrysobactin': 'C1[C@@H](C(=O)OC[C@@H](C(=O)OC[C@@H](C(=O)O1)NC(=O)[C@@H](CCCCN)NC(=O)C2=C(C(=CC=C2)O)O)NC(=O)[C@@H](CCCCN)NC(=O)C3=C(C(=CC=C3)O)O)NC(=O)[C@@H](CCCCN)NC(=O)C4=C(C(=CC=C4)O)O',
    'mirubactin': 'NC(=NCCC[C@@H](NC(=O)C1=C(O)C(O)=CC=C1)C(=O)N[C@H](CCCN(O)C=O)C(=O)O)NC(=O)C1=C(O)C(O)=CC=C1'}

mol_dict = {k:Chem.MolFromSmiles(v) for k, v in smis.items()}
export_consistent_mol_figures(mol_dict, prefix='fig1', canvas_w=400)

TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

In [192]:
# # Get maximum common substructure
# from rdkit.Chem import rdFMCS
# res = rdFMCS.FindMCS(list(mol_dict.values())[:3])
# mcs_smarts = res.smartsString
# mcs_mol = Chem.MolFromSmarts(res.smartsString)

export_consistent_mol_figures({'mcs': mcs_mol}, fixed_font_size=14, canvas_w=250)

Saved figures/mcs.svg


In [179]:
import sys
sys.path.append("/home/mattia/geass")
from new_geass import Geass
geass = Geass()

frags = []
for idx, mol in enumerate(mol_dict.values(), start=1):
    suffix = chr(ord('a') + idx - 1)
    frags.extend(geass.get_fragments(mol))
    # geass.show_connection_graph()
    # geass.export_connection_graph(f'figures/fig1{suffix}-graph.svg')

# Keep one fragment per MUID
unique_frags = {}
for frag in frags:
    if frag.HasProp('MUID'):
        key = frag.GetProp('MUID')
    unique_frags[key] = frag

export_consistent_mol_figures(unique_frags, prefix='frag', 
    fixed_font_size=5, canvas_w=50)

Saved figures/frag_C00065.svg
Saved figures/frag_C00196.svg
Saved figures/frag_C00221.svg
Saved figures/frag_C00739.svg
Saved figures/frag_C20850.svg
Saved figures/frag_C00792.svg


In [ ]:
cluster_graphs = []
for mol in mols[0:3]:
    geass.get_fragments(mol)
    g = geass.create_connection_graph()
    cluster_graphs.append(g)

import variation_graph as vg
V = vg.build_variation_graph(cluster_graphs)
vg.show_variation_graph(V)